# 05 — Final Fit & Model Export
### NutriFit-AI

Refits the selected model on **100 % of the data** and exports the artefacts the
FastAPI service loads.

Why refit on everything: the 80/20 split existed to produce an honest
performance estimate, and that estimate is already recorded in notebook 04. The
shipped model should see every available row — with 973 records, discarding 195
of them at deployment would be wasteful.

**Runtime:** CPU. Under a minute.

In [ ]:
# ============================================================
# SETUP - run this first in every notebook
# ============================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/NutriFit-AI")
    if not PROJECT.exists():
        raise FileNotFoundError(
            f"{PROJECT} not found.\n"
            "Upload the whole NutriFit-AI folder to the ROOT of your Google Drive "
            "(My Drive/NutriFit-AI), then re-run this cell."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-learn>=1.4", "pandas>=2.1", "joblib>=1.3", "seaborn>=0.13"],
        check=False,
    )
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "ml" / "nutrifit").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT / "ml"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nutrifit
from nutrifit import config, data, foods, labels, nutrition, planner, preprocessing, recommender, training

for directory in (config.PROCESSED_DIR, config.ARTIFACTS_DIR, config.FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"nutrifit  {nutrifit.__version__}")
print(f"project   {PROJECT}")
print(f"data/raw  {config.RAW_DIR}")
print(f"figures   {config.FIGURES_DIR}")
print(f"in colab  {IN_COLAB}")

## 1. Load data and select the winning model per target

Selection is by **cross-validated MAE**, not the single test split — a single
split is too noisy to choose on.

In [ ]:
import joblib, json, platform, sklearn
from datetime import datetime, timezone

labelled = pd.read_csv(config.USERS_PROCESSED)
bundles = {
    "calorie_target": joblib.load(config.ARTIFACTS_DIR / "_calorie_bundle.pkl"),
    "protein_target": joblib.load(config.ARTIFACTS_DIR / "_protein_bundle.pkl"),
}

selected = {}
for target, bundle in bundles.items():
    winner = min(bundle["results"], key=lambda k: bundle["results"][k].cv_metrics["mae"]["mean"])
    selected[target] = winner
    print(f"{target:16s} -> {winner}")
    for name, result in bundle["results"].items():
        marker = "  <-- selected" if name == winner else ""
        print(f"    {name:20s} CV MAE = {result.cv_metrics['mae']['mean']:8.3f} "
              f"+/- {result.cv_metrics['mae']['std']:.3f}{marker}")

## 2. Refit on the full dataset and export

In [ ]:
MODEL_FILES = {"calorie_target": config.CALORIE_MODEL_FILE,
               "protein_target": config.PROTEIN_MODEL_FILE}

X_full = preprocessing.select_features(labelled)

for target, bundle in bundles.items():
    pipeline = bundle["pipelines"][selected[target]]
    pipeline.fit(X_full, labelled[target].astype(float))
    path = config.ARTIFACTS_DIR / MODEL_FILES[target]
    joblib.dump(pipeline, path, compress=3)
    print(f"{target:16s} -> {path.name}  ({path.stat().st_size/1024:.1f} KB)")

## 3. Export the recommendation engine

In [ ]:
catalogue = foods.add_derived_features(pd.read_csv(config.FOODS_PROCESSED))
engine = recommender.MealRecommender(catalogue, random_state=config.RANDOM_SEED)
path = config.ARTIFACTS_DIR / config.RECOMMENDER_FILE
joblib.dump(engine, path, compress=3)
print(f"recommender -> {path.name}  ({path.stat().st_size/1024:.1f} KB)")

## 4. Reload sanity check

**Never trust `joblib.dump` without reloading.** This catches version mismatches
and missing custom transformers at build time rather than during a live demo.

In [ ]:
reference = pd.DataFrame([{
    "age": 28, "height_cm": 178.0, "weight_kg": 82.0, "bmi": 82.0/(1.78**2),
    "body_fat_pct": 18.0, "workout_frequency": 4, "session_duration_h": 1.25,
    "experience_level": 2, "gender": "Male", "fitness_goal": "muscle_gain",
    "activity_level": "active",
}])
missing = [c for c in preprocessing.FEATURE_COLUMNS if c not in reference.columns]
assert not missing, f"reference profile missing {missing}"

EXPECTED = {config.CALORIE_MODEL_FILE: (1200, 5000), config.PROTEIN_MODEL_FILE: (50, 300)}
for filename, (low, high) in EXPECTED.items():
    reloaded = joblib.load(config.ARTIFACTS_DIR / filename)
    value = float(reloaded.predict(reference[preprocessing.FEATURE_COLUMNS])[0])
    status = "OK" if low <= value <= high else "OUT OF RANGE"
    print(f"{filename:22s} -> {value:8.1f}   [{status}]")
    assert low <= value <= high, f"{filename} predicted {value}, outside [{low}, {high}]"

reloaded_engine = joblib.load(config.ARTIFACTS_DIR / config.RECOMMENDER_FILE)
top = reloaded_engine.recommend("breakfast", 2400, 150, "muscle_gain", top_n=3)
print("\nTop breakfast suggestions:")
for item in top:
    print(f"  {item.name:38s} {item.calories:6.0f} kcal  {item.protein_g:5.1f} g protein")
assert top, "recommender returned nothing"

print("\nCompare against the pure formula:")
print(nutrition.formula_targets(weight_kg=82, height_cm=178, age=28, gender="Male",
                                goal="muscle_gain", workout_frequency=4,
                                session_duration_h=1.25, experience_level=2,
                                body_fat_pct=18.0))

## 5. Write the metrics file and model card

In [ ]:
metrics = {
    "model_version": config.MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "n_rows": int(len(labelled)),
    "features": preprocessing.FEATURE_COLUMNS,
    "seed": config.RANDOM_SEED,
    "targets": {},
}
for target, bundle in bundles.items():
    metrics["targets"][target] = {
        "selected_model": selected[target],
        "refit_on_full_data": True,
        "formula_baseline": training.formula_baseline_metrics(labelled, target),
        "theoretical_r2_ceiling": labels.theoretical_r2_ceiling(labelled, target),
        "models": {name: result.to_dict() for name, result in bundle["results"].items()},
    }
(config.ARTIFACTS_DIR / config.METRICS_FILE).write_text(
    json.dumps(metrics, indent=2, default=str), encoding="utf-8")

model_card = {
    "name": "NutriFit-AI calorie & protein requirement models",
    "version": config.MODEL_VERSION,
    "trained_at": metrics["trained_at"],
    "algorithms": ["LinearRegression", "RandomForestRegressor"],
    "selected": selected,
    "features": preprocessing.FEATURE_COLUMNS,
    "training_rows": int(len(labelled)),
    "label_construction": (
        "Katch-McArdle BMR where body fat is measured, else Mifflin-St Jeor; "
        "FAO/WHO PAL multiplier; goal-specific energy adjustment and ISSN protein "
        "coefficient sampled within published ranges; Gaussian residual noise. "
        "See ml/nutrifit/labels.py."
    ),
    "intended_use": (
        "Nutritional guidance for healthy adult gym users. Not a medical device; "
        "not for clinical populations, pregnancy, or eating disorders."
    ),
    "known_limitations": [
        "Trained on 973 records; Random Forest is data-starved at this scale.",
        "Fitness goals are synthetically assigned, not self-reported.",
        "No allergy, medical-condition or dietary-restriction handling.",
        "Body fat is Deurenberg-estimated when not user-supplied.",
    ],
    "environment": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
(config.ARTIFACTS_DIR / config.MODEL_CARD_FILE).write_text(
    json.dumps(model_card, indent=2), encoding="utf-8")

print("Wrote model_metrics.json and model_card.json")
for f in sorted(config.ARTIFACTS_DIR.glob("*")):
    if not f.name.startswith("_"):
        print(f"  {f.name:24s} {f.stat().st_size/1024:8.1f} KB")

## 6. Download the artefacts

If you ran this in Colab with the project on Drive, the files are **already
saved** to `MyDrive/NutriFit-AI/ml/artifacts/` — just sync that folder back to
your laptop and skip this cell.

Otherwise use the download cell below.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for name in (config.CALORIE_MODEL_FILE, config.PROTEIN_MODEL_FILE,
                 config.RECOMMENDER_FILE, config.METRICS_FILE, config.MODEL_CARD_FILE):
        path = config.ARTIFACTS_DIR / name
        if path.exists():
            files.download(str(path))
else:
    print("Local run - artefacts are already in", config.ARTIFACTS_DIR)

## 7. Final step — on your laptop

Copy the artefacts into the FastAPI service and verify them:

```bash
python ml/scripts/export_models.py
```

That reloads every artefact, runs a prediction, and range-checks the result
before letting the copy succeed.

Then start the service:

```bash
cd services/ml-service
uvicorn app.main:app --reload --port 8000
```

---

### ⚠️ Finished with Colab? Free your compute units

`Runtime → Disconnect and delete runtime`

Closing the browser tab does **not** stop the session. An idle runtime keeps
consuming compute units until it times out.